# 01 — Exploración de Datos y Verdad Terreno

**Objetivo:** Analizar la estructura de las planillas originales en `data/raw/` (formatos PDF/imagen, resolución y cantidad de páginas) y caracterizar el dataset de Verdad Terreno en `data/processed/` (distribución de puntajes totales, categorías FATARCO/World Archery y frecuencias de valores de flecha).

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

try:
    import pypdfium2 as pdfium
except ImportError:
    pdfium = None

RAW_DIR = '../data/raw'
PROCESSED_DIR = '../data/processed'

archivos_raw = sorted(glob.glob(os.path.join(RAW_DIR, '*')))
inventario = []
for ruta in archivos_raw:
    ext = os.path.splitext(ruta)[1].lower()
    size_kb = round(os.path.getsize(ruta) / 1024, 1)
    if ext == '.pdf' and pdfium is not None:
        doc = pdfium.PdfDocument(ruta)
        n_pags = len(doc)
        w, h = doc[0].get_size() if n_pags > 0 else (0, 0)
        inventario.append({'archivo': os.path.basename(ruta), 'tipo': 'PDF', 'paginas': n_pags, 'dimensiones': f'{int(w)}x{int(h)} pt', 'tamano_kb': size_kb})
    elif ext in ['.png', '.jpg', '.jpeg']:
        with Image.open(ruta) as im:
            w, h = im.size
        inventario.append({'archivo': os.path.basename(ruta), 'tipo': ext.upper(), 'paginas': 1, 'dimensiones': f'{w}x{h} px', 'tamano_kb': size_kb})

df_inv = pd.DataFrame(inventario)
print(f'Total de documentos en {RAW_DIR}: {len(df_inv)}')
display(df_inv)

In [ ]:
csv_files = sorted(glob.glob(os.path.join(PROCESSED_DIR, '*.csv')))
print('Datasets CSV disponibles:', [os.path.basename(c) for c in csv_files])

csv_gt = csv_files[0] if csv_files else '../data/processed/torneo_indoor_san_francisco_detallado.csv'
df_gt = pd.read_csv(csv_gt)

print(f'Dataset cargado: {os.path.basename(csv_gt)} | Filas: {df_gt.shape[0]} | Columnas: {df_gt.shape[1]}')
display(df_gt.head(10))
if 'Total_Score' in df_gt.columns:
    display(df_gt[['Total_Score']].describe().T)

In [ ]:
cols_flechas = [c for c in df_gt.columns if '_' in c and c.split('_')[0].isdigit()]
if cols_flechas:
    valores_flechas = df_gt[cols_flechas].astype(str).values.ravel()
    valores_flechas = [v.strip().upper() for v in valores_flechas if v.strip().upper() not in ['NAN', 'NONE', '']]
    conteo_flechas = pd.Series(valores_flechas).value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    conteo_flechas.plot(kind='bar', ax=axes[0], color='#0284c7', edgecolor='black')
    axes[0].set_title('Distribución de Valores por Flecha (Verdad Terreno)')
    axes[0].set_xlabel('Valor anotado')
    axes[0].set_ylabel('Frecuencia')

    if 'Total_Score' in df_gt.columns:
        df_gt['Total_Score'].dropna().plot(kind='hist', bins=12, ax=axes[1], color='#10b981', edgecolor='black')
        axes[1].set_title('Histograma de Puntaje Total por Planilla')
        axes[1].set_xlabel('Puntaje Total')

    plt.tight_layout()
    plt.show()